## 32 basque-spanish words and gender preferences.


In [4]:
import torch
from transformer_lens import HookedTransformer
import pandas as pd
from tqdm import tqdm

# Load model
print("Loading model...")
model = HookedTransformer.from_pretrained("gemma-2b")
model.set_use_attn_result(True)
model.to("cuda" if torch.cuda.is_available() else "cpu")
print("Model loaded.")

Loading model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.95 GiB. GPU 0 has a total capacity of 79.14 GiB of which 5.38 MiB is free. Process 2411304 has 25.94 GiB memory in use. Process 2420401 has 47.85 GiB memory in use. Including non-PyTorch memory, this process has 5.32 GiB memory in use. Of the allocated memory 4.91 GiB is allocated by PyTorch, and 1.45 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [3]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Wed Apr 16 05:17:13 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:4B:00.0 Off |                    0 |
| N/A   27C    P0             70W /  500W |   81032MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:

basque_items = [
    ("gona", "falda", "F"), ("gazta", "queso", "M"), ("tanta", "gota", "F"), ("galtza", "pantalón", "M"),
    ("soka", "cuerda", "F"), ("uda", "verano", "M"), ("tipula", "cebolla", "F"), ("errota", "molino", "M"),
    ("gezi", "flecha", "F"), ("begi", "ojo", "M"), ("euri", "lluvia", "F"), ("zubi", "puente", "M"),
    ("ezti", "miel", "F"), ("ogi", "pan", "M"), ("ilargi", "luna", "F"), ("eguzki", "sol", "M"),
    ("belaun", "rodilla", "F"), ("komun", "cuarto de baño", "M"), ("txanpon", "moneda", "F"), ("kirten", "mango", "M"),
    ("bekain", "ceja", "F"), ("ezpain", "labio", "M"), ("babarrun", "alubia", "F"), ("eraztun", "anillo", "M"),
    ("leiho", "ventana", "F"), ("beso", "brazo", "M"), ("hosto", "hoja", "F"), ("lepo", "cuello", "M"),
    ("malko", "lágrima", "F"), ("moko", "pico", "M"), ("pospolo", "cerilla", "F"), ("soineko", "vestido", "M")
]

In [ ]:

def get_log_prob(full_text: str, phrase_start: str, continuation: str):
    toks = model.to_tokens(full_text)[0]
    logits = model(toks[:-1])[0]
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    token_ids = toks[1:]
    return sum(log_probs[i, token_ids[i]].item() for i in range(len(token_ids)))

def get_temp0_preference(noun: str):
    prompt = (
        f"Estás organizando objetos con otra persona. Debes hablar en español, "
        f"pero usa el euskera como lengua secreta solo para decir el nombre del objeto.\n"
        f"¿Cuál de estas opciones suena más natural?\nA) el {noun}\nB) la {noun}\nRespuesta:"
    )
    toks = model.to_tokens(prompt)[0]
    logits = model(toks[:-1])[0][-1]
    next_token = torch.argmax(logits).item()
    decoded = model.to_string(torch.tensor([next_token]))
    return "el" if "el" in decoded else "la"

In [ ]:

# Run task
results = []
for noun, spanish, gender in tqdm(basque_items):
    # Log-probability method
    context_a = f"Pon el {noun} aquí."
    context_b = f"Pon la {noun} aquí."
    logprob_el = get_log_prob(context_a, "Pon el ", noun)
    logprob_la = get_log_prob(context_b, "Pon la ", noun)
    logit_diff = logprob_el - logprob_la

    # Forced-choice prompt
    temp0_pref = get_temp0_preference(noun)

    results.append({
        "Basque Noun": noun,
        "Spanish Equivalent": spanish,
        "Expected Gender": gender,
        "LogProb (el)": logprob_el,
        "LogProb (la)": logprob_la,
        "Logit Preference (el - la)": logit_diff,
        "Argmax Preference (temp=0)": temp0_pref
    })

# Save results
df = pd.DataFrame(results)
df.to_csv("gemma_gender_preference_with_context.csv", index=False)
print(df.head())